# Telegram → trading-library book export

Run this notebook in Google Colab (colab.research.google.com → File → Upload notebook, or open directly from GitHub).

It downloads document attachments (PDF/EPUB/MOBI/AZW3/DJVU/TXT/MD) from a Telegram channel into `books/`, then pushes them straight to the repo on GitHub.

**Credentials are entered interactively below and never written to this file or committed anywhere.**

In [ ]:
!pip install -q telethon

In [ ]:
import getpass

TG_API_ID = input("Telegram API ID: ").strip()
TG_API_HASH = getpass.getpass("Telegram API hash: ").strip()
TG_CHANNEL = input("Channel username (e.g. mychannel) or numeric chat ID (e.g. -1001599776120): ").strip()

In [ ]:
import asyncio
import re
from pathlib import Path

from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeFilename

BOOKS_DIR = Path("books")
SESSION_NAME = "telegram_export"
DOCUMENT_EXTENSIONS = {".pdf", ".epub", ".mobi", ".azw3", ".djvu", ".txt", ".md"}


def sanitize_filename(name: str) -> str:
    name = re.sub(r"[^\w\s.\-]", "", name).strip()
    return re.sub(r"\s+", "_", name)


def get_filename(message):
    if not message.document:
        return None
    for attr in message.document.attributes:
        if isinstance(attr, DocumentAttributeFilename):
            return attr.file_name
    return None


async def resolve_channel(client, channel_ref: str):
    if not re.fullmatch(r"-?\d+", channel_ref):
        return await client.get_entity(channel_ref)

    # Numeric IDs for private channels/supergroups need the access hash,
    # which only comes from the account's own dialog list, not a bare ID lookup.
    raw = channel_ref.lstrip("-")
    candidates = {raw, raw[3:] if raw.startswith("100") else raw}
    async for dialog in client.iter_dialogs():
        if str(dialog.entity.id) in candidates:
            return dialog.entity
    raise ValueError(
        f"Could not find a channel/chat with ID {channel_ref} in your dialogs. "
        "Make sure you (this account) are a member of the channel, or use its "
        "@username instead of a numeric ID."
    )


async def export():
    BOOKS_DIR.mkdir(exist_ok=True)
    client = TelegramClient(SESSION_NAME, TG_API_ID, TG_API_HASH)
    await client.start()

    entity = await resolve_channel(client, TG_CHANNEL)

    downloaded = 0
    async for message in client.iter_messages(entity):
        filename = get_filename(message)
        if not filename:
            continue
        ext = Path(filename).suffix.lower()
        if ext not in DOCUMENT_EXTENSIONS:
            continue

        dest = BOOKS_DIR / sanitize_filename(filename)
        if dest.exists():
            continue

        print(f"Downloading {filename} -> {dest}")
        await client.download_media(message, file=str(dest))
        downloaded += 1

    print(f"Done. Downloaded {downloaded} new file(s) into {BOOKS_DIR}/")
    await client.disconnect()


await export()

## Push the books straight to GitHub

Avoids downloading/uploading a large zip. Needs a GitHub **Personal Access Token** with `repo` scope (Settings → Developer settings → Personal access tokens → Fine-grained or classic) — generate one, paste it when prompted below (hidden input, not saved anywhere), and it's only used for this push.

In [ ]:
from pathlib import Path

LIMIT_MB = 95  # GitHub hard-rejects files over 100MB via normal git push
oversized = []
for f in Path("books").iterdir():
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        if size_mb > LIMIT_MB:
            oversized.append((f.name, round(size_mb, 1)))

if oversized:
    print("These files exceed GitHub's 100MB push limit and will need Git LFS or another host:")
    for name, size in oversized:
        print(f"  - {name}: {size} MB")
else:
    print("All files are under the GitHub push size limit — safe to push directly.")

In [ ]:
import getpass
import shutil
import subprocess
from pathlib import Path

GITHUB_OWNER = "raiaashish-code"
GITHUB_REPO = "trading-library"

gh_token = getpass.getpass("GitHub Personal Access Token (repo scope): ").strip()
gh_user = input("Git commit author name (e.g. your GitHub username): ").strip()
gh_email = input("Git commit author email: ").strip()

repo_dir = Path("repo_push")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

remote = f"https://{gh_token}@github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"


def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


run(["git", "clone", "--depth", "1", remote, str(repo_dir)])
run(["git", "config", "user.name", gh_user], cwd=repo_dir)
run(["git", "config", "user.email", gh_email], cwd=repo_dir)

dest_books = repo_dir / "books"
dest_books.mkdir(exist_ok=True)
for f in Path("books").iterdir():
    if f.is_file():
        shutil.copy2(f, dest_books / f.name)

run(["git", "add", "books/"], cwd=repo_dir)

status = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo_dir)
if status.returncode == 0:
    print("No new/changed files to push.")
else:
    run(["git", "commit", "-m", "Add books exported from Telegram channel"], cwd=repo_dir)
    run(["git", "push"], cwd=repo_dir)
    print("Pushed successfully.")

# Wipe the token out of the remote URL config so it isn't left on disk
run(["git", "remote", "set-url", "origin", f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"], cwd=repo_dir)